In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/zalo-dataset/corpus.csv
/kaggle/input/zalo-dataset/crawled_test_5k.jsonl


In [2]:
!pip install protobuf==3.20.3
!pip install sentence-transformers==4.1.0 
!pip install faiss-gpu-cu12==1.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-me

In [3]:
import json
import pandas as pd
import numpy as np
import faiss
import networkx as nx
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

2026-01-14 07:19:21.448112: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768375161.835052      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768375161.953413      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
CORPUS_PATH = "/kaggle/input/zalo-dataset/corpus.csv"
TEST_PATH   = "/kaggle/input/zalo-dataset/crawled_test_5k.jsonl"
MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"
EMBED_BATCH = 512
K_GRAPH = 10
RECALL_KS = [1, 5, 10]

In [5]:
def load_corpus_csv(path):
    df = pd.read_csv(path)
    df["law_id"] = df["law_id"].str.lower().str.replace("đ", "d")
    docs = df.to_dict(orient="records")
    return docs

In [6]:
def load_test_jsonl(path):
    tests = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            q = obj.get("question", "").strip()
            rels = obj.get("relevant_articles", [])
            
            if q and rels:
                tests.append({
                    "query": q, 
                    "relevant_articles": rels  # Keep full structure
                })
    return tests

In [7]:
def embed(model, texts, batch_size=EMBED_BATCH):
    all_embs = []
    all_embs = model.encode(
        texts, 
        batch_size=batch_size, 
        convert_to_numpy=True, 
        show_progress_bar=True 
    )
    return all_embs

In [8]:
def build_hierarchical_graph(docs, embeddings, k):
    """
    Builds hierarchical graph: Field -> Law -> Article -> Khoan -> Diem
    với Field được lấy từ cột 'field' trong CSV
    """
    G = nx.Graph()
    #seen_fields = set()
    seen_laws = set()
    seen_articles = set()
    seen_khoans = set()
    
    print("Building hierarchical graph (Field -> Law -> Article -> Khoan -> Diem)...")
    
    for i, doc in enumerate(tqdm(docs, desc="Building hierarchy")):
        
        # 1. Create the Diem (text) Node
        diem_node_id = str(doc["id"])
        
        G.add_node(
            diem_node_id,
            type="Diem",
            title=doc.get("title", ""),
            text=doc.get("text", ""),
            law_id=doc.get("law_id"),
            article_id=doc.get("article_id"),
            khoan_id=doc.get("khoan_id"),
            diem_id=doc.get("diem_id"),
            #field=doc.get("field", "Khác")
        )
        
        # 2. Create and link Khoan node
        if doc.get("khoan_id") and not pd.isna(doc.get("khoan_id")):
            khoan_node_id = f"{doc['law_id']}_{doc['article_id']}_{doc['khoan_id']}"
            
            if khoan_node_id not in seen_khoans:
                G.add_node(
                    khoan_node_id,
                    type="Khoan",
                    label=f"Khoản {doc['khoan_id']}"
                )
                seen_khoans.add(khoan_node_id)
            
            G.add_edge(khoan_node_id, diem_node_id, relation="CONTAINS")
            
            # 3. Create and link Article node
            if doc.get("article_id") and not pd.isna(doc.get("article_id")):
                article_node_id = f"{doc['law_id']}_{doc['article_id']}"
                
                if article_node_id not in seen_articles:
                    G.add_node(
                        article_node_id,
                        type="Article",
                        label=f"Điều {doc['article_id']}"
                    )
                    seen_articles.add(article_node_id)
                
                G.add_edge(article_node_id, khoan_node_id, relation="CONTAINS")
                
                # 4. Create and link Law node
                if doc.get("law_id") and not pd.isna(doc.get("law_id")):
                    law_node_id = str(doc["law_id"])
                    
                    if law_node_id not in seen_laws:
                        G.add_node(
                            law_node_id,
                            type="Law",
                            label=f"Law {doc['law_id']}"
                        )
                        seen_laws.add(law_node_id)
                    
                    G.add_edge(law_node_id, article_node_id, relation="CONTAINS")
    
    print(f"Created {len(seen_laws)} Law nodes")
    print(f"Created {len(seen_articles)} Article nodes")
    print(f"Created {len(seen_khoans)} Khoan nodes")

    vectors = embeddings.copy()
    faiss.normalize_L2(vectors)
    global_index = faiss.IndexFlatIP(vectors.shape[1])
    global_index.add(vectors)
    
    print(f"\nAdding SIMILAR_TO edges (k={k}) within each Field...")
    similar_edges_count = 0
    
    for i, doc in enumerate(tqdm(docs, desc="Adding similarity edges")):
        node_id_i = str(doc["id"])
    
        # Query vector
        vec = vectors[i].reshape(1, -1)
        
        # search (k + 1) results globally
        D, I = global_index.search(vec, k + 1)
    
        for local_j, sim in zip(I[0][1:], D[0][1:]):  # skip itself (index 0)
            global_j = local_j # In global index, local_j IS the global index
            node_id_j = str(docs[global_j]["id"])
    
            if not G.has_edge(node_id_i, node_id_j):
                G.add_edge(
                    node_id_i,
                    node_id_j,
                    relation="SIMILAR_TO",
                    weight=float(sim)
                )
                similar_edges_count += 1
    
    print(f"Added {similar_edges_count} SIMILAR_TO edges (global)")
    
    return G

In [9]:
def build_faiss(embs):
    d = embs.shape[1]
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(d)
    index.add(embs)
    return index

def recall_at_k_graph(
    index,          # FAISS index
    G,              # Graph
    corpus_docs,    # List of document dictionaries (with law_id, article_id, etc.)
    corpus_ids,     # List of document IDs
    id2idx,         # Map from ID -> FAISS index
    q_embs,         # Query embeddings
    ground_truths,  # List of relevant_articles for each query
    ks,             # [1, 10, 30]
    expansion_k=20
):
    faiss.normalize_L2(q_embs)
    D, I = index.search(q_embs, expansion_k) 
    
    hits = {k: 0 for k in ks}
    total = len(q_embs)
    
    for qi in range(total):
        # Build ground truth set of (law_id, article_id) tuples
        gt_law_article_pairs = set()
        for rel_article in ground_truths[qi]:
            law_id = rel_article.get("law_id", "").lower().replace("đ", "d")
            article_id = str(rel_article.get("article_id", ""))
            if law_id and article_id:
                gt_law_article_pairs.add((law_id, article_id))
        
        if not gt_law_article_pairs:
            continue
        
        for k in ks:
            # 1. Get top-k seed documents from FAISS
            current_seed_indices = set(I[qi, :k].tolist())
            current_seed_node_ids = {corpus_ids[idx] for idx in current_seed_indices if idx < len(corpus_ids)}
            
            # 2. Expand via graph neighbors
            final_expanded_set_at_k = set(current_seed_node_ids)
            for node_id in current_seed_node_ids:
                if node_id in G:
                    final_expanded_set_at_k.update(set(G.neighbors(node_id)))
            
            # 3. Convert expanded node IDs back to document indices
            final_pred_indices_at_k = {id2idx[nid] for nid in final_expanded_set_at_k if nid in id2idx}
            
            # 4. Check if ANY retrieved document matches (law_id, article_id)
            found_match = False
            for pred_idx in final_pred_indices_at_k:
                if pred_idx < len(corpus_docs):
                    doc = corpus_docs[pred_idx]
                    doc_law_id = str(doc.get("law_id", "")).lower().replace("đ", "d")
                    doc_article_id = str(doc.get("article_id", ""))
                    
                    # Check if this document matches any ground truth pair
                    if (doc_law_id, doc_article_id) in gt_law_article_pairs:
                        found_match = True
                        break
            
            if found_match:
                hits[k] += 1
                
    return {k: hits[k]/total for k in ks}

In [10]:
print("\nLoading corpus CSV...")
corpus = load_corpus_csv(CORPUS_PATH)
print(f"Corpus size: {len(corpus):,}")

corpus_texts = [f"{d['title']} {d['text']}" for d in corpus]
corpus_ids = [str(d["id"]) for d in corpus]

# Create map id -> index
id2idx = {id_: i for i, id_ in enumerate(corpus_ids)}

# print("\nLoading test set...")
# tests = load_test_jsonl(TEST_PATH)
# print(f"Test queries: {len(tests):,}")

# queries = [t["query"] for t in tests]
# ground_truths = [t["relevant_articles"] for t in tests]

print("\nLoading embedding model...")
model = SentenceTransformer(MODEL_NAME)

print("\nEmbedding corpus...")
corpus_emb = embed(model, corpus_texts)
print(f"Embedding shape: {corpus_emb.shape}")

print("\nBuilding FAISS index...")
index = build_faiss(corpus_emb)

print("\n" + "="*70)
print("BUILDING HIERARCHICAL GRAPH (NO FIELD)")
print("="*70)
# CALL THE MODIFIED FUNCTION HERE
G = build_hierarchical_graph(corpus, corpus_emb, k=K_GRAPH)

print(f"\n{'='*70}")
print("GRAPH STATISTICS")
print("="*70)
print(f"Total nodes: {G.number_of_nodes():,}")
print(f"Total edges: {G.number_of_edges():,}")

# Count nodes by type
node_types = {}
for node in G.nodes():
    node_type = G.nodes[node].get('type', 'Unknown')
    node_types[node_type] = node_types.get(node_type, 0) + 1

print("\nNode type distribution:")
# Removed 'Field' from this list
for node_type in ['Law', 'Article', 'Khoan', 'Diem']: 
    count = node_types.get(node_type, 0)
    print(f"  {node_type:15s}: {count:,} nodes")

# Count edges by type
edge_types = {}
for u, v, data in G.edges(data=True):
    edge_type = data.get('relation', 'Unknown')
    edge_types[edge_type] = edge_types.get(edge_type, 0) + 1

print("\nEdge type distribution:")
for edge_type, count in edge_types.items():
    print(f"  {edge_type:15s}: {count:,} edges")

# print("\n" + "="*70)
# print("EMBEDDING QUERIES")
# print("="*70)
# q_emb = embed(model, queries)
# print(f"Query embeddings shape: {q_emb.shape}")

# print("\n" + "="*70)
# print("EVALUATING RECALL@K WITH GRAPH")
# print("="*70)
# recall = recall_at_k_graph(
#     index,
#     G,
#     corpus,
#     corpus_ids,
#     id2idx,
#     q_emb,
#     ground_truths,
#     RECALL_KS
# )

# print("\nResults:")
# for k in RECALL_KS:
#     print(f"  Recall@{k:2d}: {recall[k]:.4f}")

# print("\n" + "="*70)
# print("EVALUATION COMPLETE!")
# print("="*70)


Loading corpus CSV...


/tmp/ipykernel_20/1488424063.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Corpus size: 170,605

Loading embedding model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

...

Embedding shape: (170605, 768)

Building FAISS index...

BUILDING HIERARCHICAL GRAPH (NO FIELD)
Building hierarchical graph (Field -> Law -> Article -> Khoan -> Diem)...


Building hierarchy: 100%|██████████| 170605/170605 [00:02<00:00, 74034.87it/s]


Created 5275 Law nodes
Created 82004 Article nodes
Created 135931 Khoan nodes

Adding SIMILAR_TO edges (k=10) within each Field...


Adding similarity edges: 100%|██████████| 170605/170605 [2:26:53<00:00, 19.36it/s]


Added 1181506 SIMILAR_TO edges (global)

GRAPH STATISTICS
Total nodes: 393,769
Total edges: 1,551,733

Node type distribution:
  Law            : 5,275 nodes
  Article        : 81,958 nodes
  Khoan          : 135,931 nodes
  Diem           : 170,605 nodes

Edge type distribution:
  SIMILAR_TO     : 1,181,506 edges
  CONTAINS       : 370,227 edges


In [11]:
test_queries = [
    # Câu 1 (Đất đai - Giấy tờ)
    "Ông A mua đất bằng giấy viết tay năm 2003, nhưng đến năm 2024 mới làm thủ tục cấp sổ đỏ thì có được công nhận không? Việc này chịu sự điều chỉnh của Luật Đất đai 2003 hay Luật Đất đai 2024?",
    
    # Câu 2 (Hình sự - Độ tuổi)
    "Một người 17 tuổi điều khiển xe máy dung tích trên 50cm3 gây tai nạn chết người. Người này có phải chịu trách nhiệm hình sự không và cha mẹ có phải bồi thường dân sự thay không?",
    
    # Câu 3 (Hôn nhân - Tài sản)
    "Vợ chồng ly hôn, người vợ đòi chia đôi căn nhà do bố mẹ chồng tặng cho riêng người chồng trong thời kỳ hôn nhân. Tài sản này được tính là chung hay riêng?",
    
    # Câu 4 (Lao động - Thai sản)
    "Công ty có được quyền đơn phương chấm dứt hợp đồng với nhân viên nữ đang mang thai với lý do tái cơ cấu doanh nghiệp không?",
    
    # Câu 5 (Dân sự - Phạt cọc)
    "Hợp đồng đặt cọc mua nhà không ghi điều khoản phạt cọc. Nếu bên bán hủy kèo không bán nữa thì bên mua có được đòi đền bù gấp đôi số tiền cọc theo luật dân sự không?"
]

print(f"BẮT ĐẦU CHẠY THỬ NGHIỆM TRÊN {len(test_queries)} CÂU HỎI SUY LUẬN")
print("="*90)

for i, query in enumerate(test_queries):
    print(f"\n\nCÂU HỎI SỐ {i+1}: {query}")
    
    # BƯỚC 1: VECTOR SEARCH (FAISS) 
    # Embedding câu hỏi hiện tại
    q_emb = embed(model, [query])
    faiss.normalize_L2(q_emb)
    
    # Tìm Top-5 Seed Nodes
    k_seed = 5
    D, I = index.search(q_emb, k_seed)
    seed_indices = I[0]
    
    found_seed_nodes = []
    print(f"[VECTOR] Tìm thấy {len(seed_indices)} văn bản gốc (Seeds):")
    
    for idx in seed_indices:
        if idx < len(corpus):
            doc = corpus[idx]
            # Lưu lại ID để dùng cho bước Graph
            doc_id = str(doc['id'])
            found_seed_nodes.append(doc_id)
            
            # In thông tin vắn tắt
            print(f"- [ID: {doc_id}] {doc.get('title', 'No Title')}")
            print(f"\"{doc.get('text', '')[:100]}...\"")
            
    # BƯỚC 2: GRAPH EXPANSION 
    print(f"\n[GRAPH] Đang mở rộng quan hệ từ {len(found_seed_nodes)} seeds...")
    
    expanded_nodes_info = [] # Lưu trữ để in sau
    checked_nodes = set(found_seed_nodes) # Tránh trùng lặp với seed
    
    for node_id in found_seed_nodes:
        if node_id in G:
            # Lấy hàng xóm
            neighbors = list(G.neighbors(node_id))
            
            for nb in neighbors:
                if nb not in checked_nodes and nb in id2idx:
                    # Lấy thông tin cạnh quan hệ
                    edge_data = G.get_edge_data(node_id, nb)
                    relation = edge_data.get('relation', 'UNKNOWN')
                    weight = edge_data.get('weight', 0.0)
                    
                    # Lấy thông tin văn bản
                    nb_idx = id2idx[nb]
                    nb_doc = corpus[nb_idx]
                    
                    expanded_nodes_info.append({
                        "source": node_id,
                        "target": nb,
                        "relation": relation,
                        "title": nb_doc.get('title', ''),
                        "weight": weight
                    })
                    checked_nodes.add(nb)

    if not expanded_nodes_info:
        print("Không tìm thấy node mở rộng nào")
    else:
        print(f"-> Tìm thấy thêm {len(expanded_nodes_info)} node liên quan.")
        print(f"-> Dưới đây là 5 node mở rộng tiêu biểu:")
        
        # Sắp xếp theo weight (nếu là SIMILAR_TO) hoặc mặc định
        # Ưu tiên hiển thị các node có quan hệ 'CONTAINS' (cấu trúc luật) hoặc weight cao
        expanded_nodes_info.sort(key=lambda x: x['weight'], reverse=True)
        
        for item in expanded_nodes_info[:5]:
            print(f"      + Từ [{item['source']}] --({item['relation']})--> [{item['target']}]: {item['title']}")
            
    print("."*90)

BẮT ĐẦU CHẠY THỬ NGHIỆM TRÊN 5 CÂU HỎI SUY LUẬN


CÂU HỎI SỐ 1: Ông A mua đất bằng giấy viết tay năm 2003, nhưng đến năm 2024 mới làm thủ tục cấp sổ đỏ thì có được công nhận không? Việc này chịu sự điều chỉnh của Luật Đất đai 2003 hay Luật Đất đai 2024?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[VECTOR] Tìm thấy 5 văn bản gốc (Seeds):
- [ID: 5471] thông tư quy định chi tiết và hướng dẫn thi hành một số điều của nghị định số 23/2015/nđ-cp  ngày 16 tháng 02 năm 2015 của chính phủ về cấp bản sao từ sổ gốc, chứng thực bản sao từ bản chính, chứng thực chữ ký và chứng thực hợp đồng, giao dịch
"chứng thực chữ ký trên giấy tờ, văn bản. Chứng thực chữ ký trên Giấy ủy quyền theo quy định tại khoả..."
- [ID: 48999] thông tư hướng dẫn thực hiện một số nội dung của luật nhà ở và nghị định số 99/2015/nđ-cp  ngày 20 tháng 10 năm 2015 của chính phủ quy định chi tiết và hướng dẫn thi hành một số điều của luật nhà ở
"chuyển nhượng hợp đồng mua bán nhà ở thương mại. Trình tự, thủ tục chuyển nhượng hợp đồng mua bán nh..."
- [ID: 167740] sửa đổi, bổ sung một số điều của các nghị định về hộ tịch, hôn nhân và gia đình và chứng thực
"Sửa đổi, bổ sung một số điều của Nghị định số 158/2005/NĐ-CP ngày 27 tháng 12 năm 2005 về đăng ký và..."
- [ID: 147232] quy định chi tiết thi hành một số điều của luật 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[VECTOR] Tìm thấy 5 văn bản gốc (Seeds):
- [ID: 101337] luật sửa đổi, bổ sung một số điều của bộ luật hình sự số 100/2015/qh13
"Sửa đổi, bổ sung, bãi bỏ một số điều của Bộ luật Hình sự số 100/2015/QH13. 98. Sửa đổi, bổ sung Điều..."
- [ID: 7174] thông tư quy định chi tiết việc xét xử vụ án hình sự có người tham gia tố tụng là người dưới 18 tuổi thuộc thẩm quyền của tòa gia đình và người chưa thành niên
"Xét xử vụ án hình sự có bị cáo, người bị hại là người dưới 18 tuổi. 1. Khi xét xử vụ án hình sự quy ..."
- [ID: 168440] sửa đổi, bổ sung một số điều của nghị định số 34/2010/nđ-cp ngày 02 tháng 4 năm 2010 của chính phủ quy định xử phạt vi phạm hành chính trong lĩnh vực giao thông đường bộ
"Sửa đổi, bổ sung một số điều của Nghị định số 34/2010/NĐ-CP ngày 02 tháng 4 năm 2010 của Chính phủ q..."
- [ID: 145630] hướng dẫn áp dụng điều 150 về tội mua bán người và điều 151 về tội mua bán người dưới 16 tuổi của bộ luật hình sự
"truy cứu trách nhiệm hình sự trong một số trường hợp cụ thể. Truy c

...

[VECTOR] Tìm thấy 5 văn bản gốc (Seeds):
- [ID: 78773] luật hôn nhân và gia đình
"chấm dứt hôn nhân mục 1: ly hôn. Chia tài sản trong trường hợp vợ chồng sống chung với gia đình. 1. ..."
- [ID: 78775] luật hôn nhân và gia đình
"chấm dứt hôn nhân mục 1: ly hôn. Chia quyền sử dụng đất của vợ chồng khi ly hôn. 3. Trong trường hợp..."
- [ID: 78774] luật hôn nhân và gia đình
"chấm dứt hôn nhân mục 1: ly hôn. Chia quyền sử dụng đất của vợ chồng khi ly hôn. 1. Quyền sử dụng đấ..."
- [ID: 78771] luật hôn nhân và gia đình
"chấm dứt hôn nhân mục 1: ly hôn. Nguyên tắc giải quyết tài sản của vợ chồng khi ly hôn. 3. Tài sản c..."
- [ID: 78755] luật hôn nhân và gia đình
"quan hệ giữa vợ và chồng mục 1: quyền và nghĩa vụ về nhân thân. Chiếm hữu, sử dụng, định đoạt tài sả..."

[GRAPH] Đang mở rộng quan hệ từ 5 seeds...
-> Tìm thấy thêm 39 node liên quan.
-> Dưới đây là 5 node mở rộng tiêu biểu:
      + Từ [78755] --(SIMILAR_TO)--> [78756]: luật hôn nhân và gia đình
      + Từ [78755] --(SIMILAR_TO)-->

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[VECTOR] Tìm thấy 5 văn bản gốc (Seeds):
- [ID: 132429] lao động
"những quy định riêng đối với lao động nữ. Quyền đơn phương chấm dứt, tạm hoãn hợp đồng lao động của ..."
- [ID: 92608] nghị định quy định chi tiết một số điều của bộ luật lao động về chính sách đối với lao động nữ
"quy định cụ thể. Quyền đơn phương chấm dứt, tạm hoãn hợp đồng lao động của lao động nữ mang thai. 1...."
- [ID: 132306] lao động
"hợp đồng lao động. Quyền đơn phương chấm dứt hợp đồng lao động của người sử dụng lao động. 1. Người ..."
- [ID: 132304] lao động
"hợp đồng lao động. Quyền đơn phương chấm dứt hợp đồng lao động của người lao động. 1. Người lao động..."
- [ID: 130267] quy định chi tiết và hướng dẫn thi hành một số nội dung của bộ luật lao động
"hợp đồng lao động mục 1: giao kết hợp đồng lao động. Đơn phương chấm dứt hợp đồng lao động của người..."

[GRAPH] Đang mở rộng quan hệ từ 5 seeds...
-> Tìm thấy thêm 28 node liên quan.
-> Dưới đây là 5 node mở rộng tiêu biểu:
      + Từ [132306] --(SIMILAR_TO)-

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[VECTOR] Tìm thấy 5 văn bản gốc (Seeds):
- [ID: 95399] bộ luật dân sự
"quy định chung. Đặt cọc. 1. Đặt cọc là việc một bên (sau đây gọi là bên đặt cọc) giao cho bên kia (s..."
- [ID: 31303] nghị định quy định chi tiết một số điều và biện pháp thi hành luật thuế xuất khẩu, thuế nhập khẩu
"quy định chung. Thời hạn nộp thuế, bảo lãnh, đặt cọc số tiền thuế phải nộp. 3. Trường hợp sử dụng hì..."
- [ID: 52847] thông tư hướng dẫn việc bán cổ phần lần đầu và chuyển nhượng vốn nhà nước theo phương thức dựng sổ
"chuyển nhượng vốn nhà nước, vốn đu tư của doanh nghiệp nhà nước tại công ty cổ phn theo phương thức ..."
- [ID: 52850] thông tư hướng dẫn việc bán cổ phần lần đầu và chuyển nhượng vốn nhà nước theo phương thức dựng sổ
"quản lý tiền đặt cọc và tiền thu từ bán cổ phn. Quản lý tiền đặt cọc và thanh toán tiền mua cổ phần...."
- [ID: 120756] quy định xử phạt vi phạm hành chính trong hoạt động xây dựng; ki
"hành vi vi phạm hành chính, hình thức xử phạt và biện pháp khắc phục hậu quả trong lĩnh